# Benchmark Gravitino vs Hive Metastore

In [ ]:
pip install hdfs

In [11]:
from hdfs import InsecureClient
import os

# Create a HDFS connector client
hdfs_client = InsecureClient("http://hive:50070", user='root')

### Create Spark Session

In [1]:
import pyspark
import os
import sys
import time
from pyspark.sql import SparkSession

spark_home = "/home/jovyan/spark-3.4.2-bin-hadoop3"
gravitino_connector_jar = os.getenv('SPARK_CONNECTOR_JAR')
os.environ['HADOOP_USER_NAME']="anonymous"

spark = SparkSession.builder \
    .appName("Benchmark") \
    .config("spark.plugins", "org.apache.gravitino.spark.connector.plugin.GravitinoSparkPlugin") \
    .config("spark.jars", f"/tmp/gravitino/spark/packages/iceberg-spark-runtime-3.4_2.12-1.6.1.jar,/tmp/gravitino/spark/packages/{gravitino_connector_jar}") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.gravitino.uri", "http://gravitino:8090") \
    .config("spark.sql.gravitino.metalake", "metalake_demo") \
    .config("spark.sql.gravitino.enableIcebergSupport", "true") \
    .config("spark.sql.catalog.hive", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.hive.type", "hive") \
    .config("spark.sql.catalog.hive.uri", "thrift://hive:9083") \
    .config("spark.locality.wait.node", "0") \
    .config("spark.sql.warehouse.dir", "hdfs://hive:9000/user/hive/warehouse") \
    .enableHiveSupport() \
    .getOrCreate()

In [28]:
spark.sql("CREATE DATABASE IF NOT EXISTS hive.benchmark LOCATION 'hdfs://hive:9000/user/hive/warehouse/benchmark.db';")
spark.sql("CREATE DATABASE IF NOT EXISTS catalog_iceberg.benchmark;")

DataFrame[]

In [27]:
def benchmark_create_tables(catalog, n_tables=100):
    durations = []
    for i in range(n_tables):
        table_name = f"{catalog}.benchmark.tbl_{i}"
        start = time.time()
        spark.sql(f"CREATE TABLE IF NOT EXISTS {table_name} (id INT, name STRING) USING iceberg")
        end = time.time()
        durations.append(end - start)
        spark.sql(f"DROP TABLE {table_name}")
        if catalog == "hive":
            spark.sql
            hdfs_client.delete(f"/user/hive/warehouse/benchmark.db/tbl_{i}", recursive=True)
        if catalog == "catalog_iceberg":
            hdfs_client.delete(f"/user/iceberg/warehouse/benchmark/tbl_{i}", recursive=True)
    return durations

hms_times = benchmark_create_tables("hive", 50)
gravitino_times = benchmark_create_tables("catalog_iceberg", 50)

Py4JJavaError: An error occurred while calling o49.sql.
: org.apache.iceberg.exceptions.NotFoundException: Failed to open input stream for file: hdfs://hive:9000/user/hive/warehouse/benchmark.db/tbl_0/metadata/00000-f2c32594-2905-43b3-8b26-61457a546678.metadata.json
	at org.apache.iceberg.hadoop.HadoopInputFile.newStream(HadoopInputFile.java:185)
	at org.apache.iceberg.TableMetadataParser.read(TableMetadataParser.java:279)
	at org.apache.iceberg.TableMetadataParser.read(TableMetadataParser.java:273)
	at org.apache.iceberg.BaseMetastoreTableOperations.lambda$refreshFromMetadataLocation$0(BaseMetastoreTableOperations.java:182)
	at org.apache.iceberg.BaseMetastoreTableOperations.lambda$refreshFromMetadataLocation$1(BaseMetastoreTableOperations.java:201)
	at org.apache.iceberg.util.Tasks$Builder.runTaskWithRetry(Tasks.java:413)
	at org.apache.iceberg.util.Tasks$Builder.runSingleThreaded(Tasks.java:219)
	at org.apache.iceberg.util.Tasks$Builder.run(Tasks.java:203)
	at org.apache.iceberg.uti

In [24]:
print("hms create:", round(sum(hms_times),2),", gravitino create",round(sum(gravitino_times),2))

hms create: 1.26 , gravitino create 2.24


In [29]:
spark.sql("CREATE TABLE IF NOT EXISTS hive.benchmark.big_tbl (id INT, p INT) USING iceberg PARTITIONED BY (p)")
spark.sql("CREATE TABLE IF NOT EXISTS catalog_iceberg.benchmark.big_tbl (id INT, p INT) USING iceberg PARTITIONED BY (p)")

def benchmark_insert(catalog, insert_times):
    start = time.time()
    for i in range(insert_times):
        spark.sql(f"INSERT INTO {catalog}.benchmark.big_tbl VALUES ({i}, {i})")
    return time.time() - start

def benchmark_alter(catalog):
    start = time.time()
    spark.sql(f"ALTER TABLE {catalog}.benchmark.big_tbl ADD COLUMN new_col STRING")
    return time.time() - start

hms_insert = benchmark_insert("hive", 50)
gravitino_insert = benchmark_insert("catalog_iceberg", 50)
hms_alter = benchmark_alter("hive")
gravitino_alter = benchmark_alter("catalog_iceberg")

In [21]:
print("hms insert: ",round(hms_insert,2),", gravitino insert:",round(gravitino_insert,2))
print("hms alter: ",round(hms_alter,2),", gravitino alter:",round(gravitino_alter,2))

hms insert:  104.77 , gravitino insert: 136.41
hms alter:  0.67 , gravitino alter: 0.51


In [34]:
def benchmark_query(catalog):
    start = time.time()
    spark.sql(f"SELECT COUNT(*) FROM {catalog}.benchmark.big_tbl WHERE p BETWEEN 10 AND 30").collect()
    return time.time() - start

hms_query = benchmark_query("hive")
gravitino_query = benchmark_query("catalog_iceberg")

In [35]:
print("hms query: ",round(hms_query,2),", gravitino query:",round(gravitino_query,2))

hms query:  0.12 , gravitino query: 0.14


In [37]:
from concurrent.futures import ThreadPoolExecutor

def describe_table(catalog):
    return spark.sql(f"DESCRIBE TABLE {catalog}.benchmark.big_tbl").collect()

def concurrency_test(catalog, n_clients=100):
    with ThreadPoolExecutor(max_workers=20) as executor:
        futures = [executor.submit(describe_table, catalog) for i in range(n_clients)]
        results = [f.result() for f in futures]
    return results

hms_concurrent = concurrency_test("hive", 200)
gravitino_concurrent = concurrency_test("catalog_iceberg", 200)

In [38]:
print("hms concurrent: ",len(hms_concurrent),", gravitino concurrent:",len(gravitino_concurrent))

hms concurrent:  200 , gravitino concurrent: 200


In [45]:
spark.sql("DESCRIBE EXTENDED catalog_iceberg.benchmark.big_tbl").show(truncate=False)

+----------------------------+---------------------------------------------------------+-------+
|col_name                    |data_type                                                |comment|
+----------------------------+---------------------------------------------------------+-------+
|id                          |int                                                      |null   |
|p                           |int                                                      |null   |
|new_col                     |string                                                   |null   |
|# Partition Information     |                                                         |       |
|# col_name                  |data_type                                                |comment|
|p                           |int                                                      |null   |
|                            |                                                         |       |
|# Metadata Columns          |

In [41]:
def get_snapshots(catalog, namespace, table):
    query = f"SELECT snapshot_id, committed_at FROM {catalog}.`{namespace}.{table}.id` ORDER BY committed_at DESC"
    return spark.sql(query).collect()

def benchmark_time_travel(catalog, snapshot_id):
    start = time.time()
    spark.sql(
        f"SELECT * FROM {catalog}.benchmark.big_tbl VERSION AS OF {snapshot_id} LIMIT 10"
    ).collect()
    return round(time.time() - start, 2)

hms_snapshots = get_snapshots("hive", "benchmark", "big_tbl")
hms_tt = benchmark_time_travel("hive", hms_snapshots[0]["snapshot_id"])

gravitino_snapshots = get_snapshots("catalog_iceberg", "benchmark", "big_tbl")
gravitino_tt = benchmark_time_travel("catalog_iceberg", gravitino_snapshots[0]["snapshot_id"])

AnalysisException: Unsupported data source type for direct query on files: hive.; line 1 pos 38